# EEG outlier review and cleaning

This notebook is for the **Siena Scalp EEG Database** used by `final_project`. It never modifies `data/raw`.

EEG requires a different policy from ordinary tabular data. Large amplitudes can be seizure physiology, eye or muscle activity, movement, or electrode failure. Therefore:

- global statistical amplitude outliers are **review flags**, not automatic deletions;
- auxiliary EKG, pulse-oximetry, heart-rate, marker, and unnamed signals are excluded from the EEG dataset;
- channel validity is decided independently in short analysis windows after detrending and robust common-median referencing;
- flat, materially clipped, non-finite, or extreme post-reference channels are rejected for that window only;
- rejected samples are not winsorized or imputed, because doing so would manufacture EEG signal.

## Phase 1 — Locate sources and protect raw data

The expected recording list and preserved raw-data audit are read as source material. We also count locally available EDF files. A missing EDF source is reported explicitly rather than silently treating audit statistics as cleaned signal samples.

In [1]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from scipy import signal

pd.set_option("display.max_columns", None)

def find_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "RECORDS").exists():
            return candidate
        nested = candidate / "26-the-optimizers-analysis" / "final_project"
        if (nested / "data" / "raw" / "RECORDS").exists():
            return nested
    raise FileNotFoundError("Could not locate final_project/data/raw/RECORDS")

PROJECT_DIR = find_project_dir(Path.cwd().resolve())
RAW_DIR = PROJECT_DIR / "data" / "raw"
INTERIM_DIR = PROJECT_DIR / "data" / "interim"
AUDIT_DIR = PROJECT_DIR / "results" / "raw_data_audit"
METRICS_PATH = AUDIT_DIR / "sampled_channel_metrics.csv"
INVENTORY_PATH = AUDIT_DIR / "edf_file_inventory.csv"

expected_recordings = [line.strip() for line in (RAW_DIR / "RECORDS").read_text().splitlines() if line.strip()]
local_edfs = sorted(RAW_DIR.rglob("*.edf"))
inventory = pd.read_csv(INVENTORY_PATH)
metrics = pd.read_csv(METRICS_PATH)

print(f"Project: {PROJECT_DIR}")
print(f"Expected EDF recordings: {len(expected_recordings)}")
print(f"Locally materialized EDF recordings: {len(local_edfs)}")
print(f"Preserved channel-audit rows: {len(metrics):,}")
print(f"Header-derived source size: {inventory['file_size_mb'].sum() / 1024:.2f} GB")

Project: C:\Users\yi_li\OneDrive\cosmos\26-the-optimizers-analysis\final_project
Expected EDF recordings: 41
Locally materialized EDF recordings: 0
Preserved channel-audit rows: 1,733
Header-derived source size: 21.29 GB


## Phase 2 — Keep scalp EEG and normalize channel names

Only labels beginning with `EEG` are scalp-EEG channels. Label case varies in the source (`Fp2` versus `FP2`, `Cz` versus `CZ`), so labels are normalized to uppercase electrode codes for reliable pooling. Numeric values are validated as microvolts.

In [2]:
is_eeg = metrics["channel"].str.match(r"^EEG\s+", case=False, na=False)
eeg = metrics.loc[is_eeg].copy()
auxiliary = metrics.loc[~is_eeg].copy()

def normalize_eeg_label(label: str) -> str:
    electrode = re.sub(r"^EEG\s+", "", str(label), flags=re.IGNORECASE).strip()
    return electrode.upper()

eeg = eeg.rename(columns={"channel": "channel_original"})
eeg.insert(2, "channel_normalized", eeg["channel_original"].map(normalize_eeg_label))
numeric_columns = ["samples_sampled", "mean", "std", "rms", "p005", "p995", "max_abs", "flat_fraction", "clipped_fraction"]
nonfinite_measurement = ~np.isfinite(eeg[numeric_columns].astype(float)).all(axis=1)
wrong_unit = eeg["unit"].ne("uV")

print(f"Scalp-EEG channel/recording pairs retained for QC: {len(eeg):,}")
print(f"Auxiliary channel/recording pairs excluded: {len(auxiliary):,}")
print(f"Normalized electrode names: {eeg['channel_normalized'].nunique()}")
print(f"Non-finite metric rows: {nonfinite_measurement.sum()}")
print(f"Non-microvolt EEG rows: {wrong_unit.sum()}")

Scalp-EEG channel/recording pairs retained for QC: 1,255
Auxiliary channel/recording pairs excluded: 478
Normalized electrode names: 31
Non-finite metric rows: 0
Non-microvolt EEG rows: 0


## Phase 3 — Find outliers without deleting physiology

Three complementary screens are applied to the preserved 40-second-per-channel audit:

1. A global 1.5×IQR high-RMS flag reproduces the original audit's broad amplitude review.
2. A modified z-score on `log1p(RMS)` compares each channel with other channels in the same recording; absolute scores above 3.5 receive focused review.
3. Near-flat, materially clipped, non-finite, or wrong-unit audit rows are invalid source rows and may be excluded automatically.

A sampled absolute amplitude above 1,000 µV is marked for review only. The project's 1,000 µV rejection threshold is correctly applied later, after detrending and common-median referencing within each analysis window.

In [3]:
q1, q3 = eeg["rms"].quantile([0.25, 0.75])
iqr = q3 - q1
rms_lower = q1 - 1.5 * iqr
rms_upper = q3 + 1.5 * iqr

log_rms = np.log1p(eeg["rms"])
file_median = log_rms.groupby(eeg["file"]).transform("median")
file_mad = log_rms.groupby(eeg["file"]).transform(lambda values: np.median(np.abs(values - np.median(values))))
modified_z = pd.Series(np.where(file_mad > 0, 0.6745 * (log_rms - file_median) / file_mad, 0.0), index=eeg.index)

eeg["global_high_rms_review"] = eeg["rms"] > rms_upper
eeg["within_recording_rms_review"] = modified_z.abs() > 3.5
eeg["within_recording_modified_z"] = modified_z.round(4)
eeg["sampled_gt_1000uv_review"] = eeg["max_abs"] > 1_000.0
eeg["near_flat_invalid"] = eeg["flat_fraction"] >= 0.95
eeg["material_clipping_invalid"] = eeg["clipped_fraction"] >= 0.01
eeg["nonfinite_metrics_invalid"] = nonfinite_measurement
eeg["wrong_unit_invalid"] = wrong_unit

invalid_columns = ["near_flat_invalid", "material_clipping_invalid", "nonfinite_metrics_invalid", "wrong_unit_invalid"]
review_columns = ["global_high_rms_review", "within_recording_rms_review", "sampled_gt_1000uv_review"]
eeg["exclude_entire_channel_recording"] = eeg[invalid_columns].any(axis=1)
eeg["requires_manual_review"] = eeg[review_columns].any(axis=1)

print(f"Global high-RMS review bound: > {rms_upper:.1f} µV")
print(f"Global high-RMS review flags: {eeg['global_high_rms_review'].sum()}")
print(f"Within-recording robust review flags: {eeg['within_recording_rms_review'].sum()}")
print(f"Sampled >1,000 µV review flags: {eeg['sampled_gt_1000uv_review'].sum()}")
print(f"Invalid channel/recording rows requiring automatic exclusion: {eeg['exclude_entire_channel_recording'].sum()}")

Global high-RMS review bound: > 102.7 µV
Global high-RMS review flags: 197
Within-recording robust review flags: 83
Sampled >1,000 µV review flags: 171
Invalid channel/recording rows requiring automatic exclusion: 0


## Phase 4 — Make the cleaning decision

Statistical amplitude flags remain eligible for window-level cleaning because seizure EEG is expected to contain unusual amplitudes. Only structurally invalid channel/recording rows are excluded globally. Every retained channel is still re-evaluated inside every short window.

In [4]:
def review_reason(row: pd.Series) -> str:
    reasons = []
    if row["global_high_rms_review"]: reasons.append("global high RMS")
    if row["within_recording_rms_review"]: reasons.append("within-recording robust RMS outlier")
    if row["sampled_gt_1000uv_review"]: reasons.append("sampled absolute amplitude >1000 uV")
    if row["near_flat_invalid"]: reasons.append("near-flat")
    if row["material_clipping_invalid"]: reasons.append("material clipping")
    if row["nonfinite_metrics_invalid"]: reasons.append("non-finite audit metrics")
    if row["wrong_unit_invalid"]: reasons.append("unexpected unit")
    return "; ".join(reasons)

eeg["review_reason"] = eeg.apply(review_reason, axis=1)
eeg["cleaning_action"] = np.where(
    eeg["exclude_entire_channel_recording"],
    "exclude channel from recording",
    "retain; apply window-level QC",
)

clean_manifest = eeg.loc[~eeg["exclude_entire_channel_recording"]].copy()
outlier_review = eeg.loc[eeg["requires_manual_review"] | eeg["exclude_entire_channel_recording"]].copy()

print(f"EEG channel/recording pairs in clean manifest: {len(clean_manifest):,}")
print(f"Pairs excluded globally: {eeg['exclude_entire_channel_recording'].sum()}")
print(f"Pairs retained but prioritized for manual review: {eeg['requires_manual_review'].sum()}")
display(outlier_review[["file", "channel_original", "channel_normalized", "rms", "max_abs", "review_reason", "cleaning_action"]].head(15))

EEG channel/recording pairs in clean manifest: 1,255
Pairs excluded globally: 0
Pairs retained but prioritized for manual review: 231


,file,channel_original,channel_normalized,rms,max_abs,review_reason,cleaning_action
4,PN00\PN00-1.edf,EEG O1,O1,111.449857,529.125,global high RMS; within-recording robust RMS o...,retain; apply window-level QC
10,PN00\PN00-1.edf,EEG Cp1,CP1,104.546055,535.500,global high RMS; within-recording robust RMS o...,retain; apply window-level QC
105,PN00\PN00-4.edf,EEG Fp1,FP1,446.032757,2537.125,global high RMS; within-recording robust RMS o...,retain; apply window-level QC
121,PN00\PN00-4.edf,EEG Fp2,FP2,78.477955,445.750,within-recording robust RMS outlier,retain; apply window-level QC
191,PN01\PN01-1.edf,EEG Fp2,FP2,981.419945,1807.875,global high RMS; within-recording robust RMS o...,retain; apply window-level QC
210,PN03\PN03-1.edf,EEG Fp1,FP1,840.389020,4095.875,global high RMS; within-recording robust RMS o...,retain; apply window-level QC
212,PN03\PN03-1.edf,EEG C3,C3,144.673153,2017.500,global high RMS; sampled absolute amplitude >1...,retain; apply window-level QC
223,PN03\PN03-1.edf,EEG Fz,FZ,371.434932,3850.375,global high RMS; sampled absolute amplitude >1...,retain; apply window-level QC
225,PN03\PN03-1.edf,EEG Pz,PZ,117.306258,3358.750,global high RMS; sampled absolute amplitude >1...,retain; apply window-level QC
229,PN03\PN03-1.edf,EEG O2,O2,152.304456,1764.625,global high RMS; sampled absolute amplitude >1...,retain; apply window-level QC


## Phase 5 — Define the actual window-level EEG cleaning operation

The function below implements the project method on an array shaped `(channels, samples)`:

1. linear detrending per channel and a pre-reference flatness check;
2. robust common-median reference at each sample;
3. rejection of non-finite, nearly flat (`pre-reference SD < 0.5 µV`), extreme (`post-reference max |amplitude| > 1,000 µV`), or materially clipped (`>1%`) channels;
4. requirement of at least 10 usable EEG channels;
5. 60 Hz notch filtering where the sample rate permits it.

Rejected channels are represented by the returned boolean mask; they are not interpolated. The function is tested with a synthetic window before use.

In [5]:
MIN_STD_UV = 0.5
MAX_ABS_UV = 1_000.0
MAX_CLIPPED_FRACTION = 0.01
MIN_USABLE_CHANNELS = 10

def clean_eeg_window(
    data_uv: np.ndarray,
    sample_rate_hz: float,
    physical_min_uv: np.ndarray,
    physical_max_uv: np.ndarray,
    min_usable_channels: int = MIN_USABLE_CHANNELS,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    data_uv = np.asarray(data_uv, dtype=float)
    if data_uv.ndim != 2:
        raise ValueError("data_uv must have shape (channels, samples)")
    physical_min_uv = np.asarray(physical_min_uv, dtype=float)
    physical_max_uv = np.asarray(physical_max_uv, dtype=float)
    if physical_min_uv.shape != (data_uv.shape[0],) or physical_max_uv.shape != (data_uv.shape[0],):
        raise ValueError("Physical limits must contain one value per channel")

    detrended = signal.detrend(data_uv, axis=1, type="linear")
    pre_reference_std = np.std(detrended, axis=1)
    referenced = detrended - np.median(detrended, axis=0, keepdims=True)
    standard_deviation = np.std(referenced, axis=1)
    maximum_absolute = np.max(np.abs(referenced), axis=1)
    tolerance = np.maximum((physical_max_uv - physical_min_uv) * 0.001, 1e-6)
    clipped_fraction = np.mean(
        (data_uv <= physical_min_uv[:, None] + tolerance[:, None])
        | (data_uv >= physical_max_uv[:, None] - tolerance[:, None]), axis=1
    )
    usable = (
        np.isfinite(referenced).all(axis=1)
        & (pre_reference_std >= MIN_STD_UV)
        & (maximum_absolute <= MAX_ABS_UV)
        & (clipped_fraction <= MAX_CLIPPED_FRACTION)
    )
    qc = pd.DataFrame({
        "pre_reference_std_uv": pre_reference_std,
        "post_reference_std_uv": standard_deviation,
        "max_abs_uv": maximum_absolute,
        "clipped_fraction": clipped_fraction,
        "usable": usable,
    })
    if int(usable.sum()) < min_usable_channels:
        raise ValueError(f"Only {int(usable.sum())} channels passed QC; {min_usable_channels} required")

    cleaned = referenced[usable]
    if sample_rate_hz > 122:
        notch_b, notch_a = signal.iirnotch(60.0, 30.0, fs=sample_rate_hz)
        cleaned = signal.filtfilt(notch_b, notch_a, cleaned, axis=1)
    return cleaned.astype(np.float32), usable, qc

rng = np.random.default_rng(42)
synthetic = rng.normal(0, 20, size=(12, 5 * 512))
synthetic[0] = 0
synthetic[1, 100] = 20_000
synthetic_clean, synthetic_mask, synthetic_qc = clean_eeg_window(
    synthetic, 512.0, np.full(12, -4096.0), np.full(12, 4096.0)
)
assert synthetic_clean.shape == (10, 5 * 512)
assert not synthetic_mask[0] and not synthetic_mask[1]
assert np.isfinite(synthetic_clean).all()
print("Synthetic cleaning test passed: flat and extreme channels rejected; 10 clean channels retained.")

Synthetic cleaning test passed: flat and extreme channels rejected; 10 clean channels retained.


## Phase 6 — Write interim artifacts and report materialization status

The normalized clean channel manifest, focused outlier-review table, and run summary are written to `data/interim`. If all 41 EDF files are absent, the notebook records `signal_arrays_materialized = false`; it does not pretend that audit rows are EEG samples.

In [6]:
MANIFEST_PATH = INTERIM_DIR / "eeg_clean_channel_manifest.csv"
OUTLIER_PATH = INTERIM_DIR / "eeg_outlier_review.csv"
SUMMARY_PATH = INTERIM_DIR / "eeg_cleaning_summary.json"

manifest_columns = [
    "patient_id", "file", "channel_original", "channel_normalized", "unit",
    "samples_sampled", "mean", "std", "rms", "p005", "p995", "max_abs",
    "flat_fraction", "clipped_fraction", "global_high_rms_review",
    "within_recording_rms_review", "within_recording_modified_z",
    "sampled_gt_1000uv_review", "requires_manual_review", "review_reason",
    "cleaning_action",
]
review_columns_output = manifest_columns + invalid_columns + ["exclude_entire_channel_recording"]

summary = {
    "dataset": "Siena Scalp EEG Database 1.0.0",
    "expected_edf_recordings": len(expected_recordings),
    "local_edf_recordings": len(local_edfs),
    "source_size_gb": round(float(inventory["file_size_mb"].sum() / 1024), 2),
    "audit_channel_rows": len(metrics),
    "auxiliary_rows_excluded": len(auxiliary),
    "eeg_channel_recording_pairs": len(eeg),
    "globally_excluded_eeg_pairs": int(eeg["exclude_entire_channel_recording"].sum()),
    "clean_manifest_pairs": len(clean_manifest),
    "manual_review_pairs": int(eeg["requires_manual_review"].sum()),
    "global_high_rms_review_bound_uv": round(float(rms_upper), 4),
    "global_high_rms_flags": int(eeg["global_high_rms_review"].sum()),
    "within_recording_robust_rms_flags": int(eeg["within_recording_rms_review"].sum()),
    "sampled_gt_1000uv_flags": int(eeg["sampled_gt_1000uv_review"].sum()),
    "signal_arrays_materialized": len(local_edfs) == len(expected_recordings),
    "signal_materialization_note": (
        "All raw EDFs are locally available for chunked window cleaning."
        if len(local_edfs) == len(expected_recordings)
        else "Raw EDF files are absent; interim outputs are the cleaned channel manifest and audit, not filtered signal arrays."
    ),
}

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
clean_manifest[manifest_columns].to_csv(MANIFEST_PATH, index=False)
outlier_review[review_columns_output].to_csv(OUTLIER_PATH, index=False)
SUMMARY_PATH.write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")

assert pd.read_csv(MANIFEST_PATH).shape[0] == len(clean_manifest)
assert pd.read_csv(OUTLIER_PATH).shape[0] == len(outlier_review)
print(f"Wrote: {MANIFEST_PATH}")
print(f"Wrote: {OUTLIER_PATH}")
print(f"Wrote: {SUMMARY_PATH}")
display(pd.Series(summary).to_frame("result"))

Wrote: C:\Users\yi_li\OneDrive\cosmos\26-the-optimizers-analysis\final_project\data\interim\eeg_clean_channel_manifest.csv
Wrote: C:\Users\yi_li\OneDrive\cosmos\26-the-optimizers-analysis\final_project\data\interim\eeg_outlier_review.csv
Wrote: C:\Users\yi_li\OneDrive\cosmos\26-the-optimizers-analysis\final_project\data\interim\eeg_cleaning_summary.json


,result
dataset,Siena Scalp EEG Database 1.0.0
expected_edf_recordings,41
local_edf_recordings,0
source_size_gb,21.29
audit_channel_rows,1733
auxiliary_rows_excluded,478
eeg_channel_recording_pairs,1255
globally_excluded_eeg_pairs,0
clean_manifest_pairs,1255
manual_review_pairs,231
